# M4L4 - RNA-Seq Review

At this point, we have covered the steps of RNA-Seq multiple times. This lab acts as a reference for this information and provides context to the practices about the execution and interpretation of DE analysis on RNA-Seq data. 

### Topics covered in this Notebook: 
- RNA-seq workflow
- Experimental design considerations
- Setting up DE analysis

#### To perform Differential Gene Expression analysis, we need to start with a matrix of counts representing the levels of gene expression. It is important to understand how the count matrix is generated before diving into the statistical analysis. Below is an image of a pseudo-alignment pipeline for RNA-Seq, similar to the one we had used last week in Module 3.


```{admonition} Adapted from
:class: tip
**Source:** [Introduction to DGE](https://hbctraining.github.io/Intro-to-DGE/), Harvard Chan Bioinformatics Core (HBC) training team, [hbctraining/Intro-to-DGE](https://github.com/hbctraining/Intro-to-DGE) (lesson "Experimental planning considerations" and workflow overview).

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Figures:** unless a caption says otherwise, the diagrams and example plots in this notebook come from the same HBC lessons, CC BY 4.0, unchanged.

**Changes:** Shortened and rewritten for this course; the example dataset is the Arabidopsis iron-homeostasis series GSE79275 (opt3-2 vs Col-0); the pipeline is described as fastp + Salmon on the Hellbender HPC; third-party figures that HBC reproduces were removed.
```



## RNA-Seq Workflow
![workflow-salmon-DGE-alt2.png](images/M4L4_RNA_Seq_Review_7dc69e25.png)

The steps used in this pipeline are as follows: 
#### 1. RNA Extraction and Library Preparation

Before RNA can be sequenced, it must first be extracted and separated from its cellular environment prepared into a cDNA library. There are a number of steps involved, which are outlined in the figure below, and in parallel, there are various quality checks implemented to make sure we have good quality RNA to move forward with. 

*[Library-preparation figure omitted (third-party figure reproduced in the HBC lessons). See the [HBC Intro-to-DGE lessons](https://hbctraining.github.io/Intro-to-DGE/).]*

#### 2. Sequencing (Illumina) - Sequencing by Synthesis

Sequencing of the cDNA libraries will generate reads. Reads correspond to the nucleotide sequences of the ends of each of the cDNA fragements in the library. You will have the choice of sequencing either a single end of the cDNA fragments (single-end reads) or both ends of the fragments (paired-end reads).
- SE - Single end dataset => Only Read1
- PE - Paired-end dataset => Read1 + Read2
    - can be 2 separate FastQ files or just one with interleaved pairs

Generally, single-end sequencing is sufficient unless it is expected that the reads will match multiple locations on the genome (e.g., organisms with many paralogous genes), assemblies are being performed, or for splice isoform differentiation. Be aware that paired-end reads are generally 2x more expensive.

Illumina sequencing technology uses a sequencing-by-synthesis approach. To explore sequencing by synthesis in more depth, please watch this linked video on [Illumina’s YouTube channel](https://www.youtube.com/watch?v=fCd6B5HRaZ8).

A brief explanation of the steps is below:

- Cluster growth: The DNA fragments in the cDNA library are denatured and hybridized to the glass flowcell (adapter complementarity). Each fragment is then clonally amplified, forming a cluster of double-stranded DNA. This step is necessary to ensure that the sequencing signal will be strong enough to be detected/captured unambiguously for each base of each fragment.

- Sequencing: The sequencing of the fragment ends is based on fluorophore-labelled dNTPs with reversible terminator elements. In each sequencing cycle, a base is incorporated into every cluster and excited by a laser.

- Image acquisition: Each dNTP has a distinct excitatory signal emission, which is captured by cameras.

- Base calling: The Base calling program will then generate the sequence of bases, i.e,. reads, for each fragment/cluster by assessing the images captured during the many sequencing cycles. In addition to calling the base in every position, the base caller will also report the certainty with which it was able to make the call (quality information).
  
  *[Illumina sequencing-by-synthesis figure omitted (vendor figure). See the [HBC Intro-to-DGE lessons](https://hbctraining.github.io/Intro-to-DGE/).]*

#### 3. Quality control of raw sequencing data (`fastp`)

The raw reads obtained from the sequencer are stored as FASTQ files. The FASTQ file format is the go-to file format for sequence reads generated from next-generation sequencing technologies.

Each FASTQ file is a text file that represents sequence readouts for a sample. Each read is represented by 4 lines as shown below:

```
@HWI-ST330:304:H045HADXX:1:1101:1111:61397
CACTTGTAAGGGCAGGCCCCCTTCACCCTCCCGCTCCTGGGGGANNNNNNNNNNANNNCGAGGCCCTGGGGTAGAGGGNNNNNNNNNNNNNNGATCTTGG
+
@?@DDDDDDHHH?GH:?FCBGGB@C?DBEGIIIIAEF;FCGGI#########################################################
```
| Line | Description                                                                 |
|------|-----------------------------------------------------------------------------|
| 1    | Always begins with ‘@’ and then provides information about the read         |
| 2    | The actual DNA sequence                                                      |
| 3    | Always begins with a ‘+’ and sometimes the same info as in line 1           |
| 4    | Has a string of characters that represent the quality scores; must have the same number of characters as line 2 |

In our pipeline, we have used `fastp` to check the quality of our reads and trim adapter sequences. The QC metrics generated create an HTML file containing information on the read quality, N-content, and read length. It is important to check this before moving on to mapping, as adapter contamination or poor quality reads can impact expression levels and thereby downstream analysis.

#### 4. Mapping reads

Once we have explored the quality of our raw reads, we can move on to quantifying expression at the transcript level. The goal of this step is to identify from which transcript each of the reads originated and the total number of reads associated with each transcript.

Tools that have been found to be most accurate for this step in the analysis are referred to as lightweight alignment tools, which include:

    Kallisto,
    Sailfish and
    Salmon

Each of the tools in the list above works slightly differently from one another. However, common to all of them is that they avoid base-to-base genomic alignment of the reads. Genomic alignment is a step performed by older splice-aware alignment tools such as STAR and HISAT2. In comparison to these tools, the lightweight alignment tools not only provide quantification estimates faster (typically more than 20 times faster), but also prove improvements in accuracy [1].

We will use the expression estimates, often referred to as ‘pseudocounts’, obtained from Salmon as the starting point for the differential gene expression analysis. We used `salmon` in Module 3 for our pseudo-alignment. 

#### 5. DE Analysis and Functional Analysis

After further QC (optional) with `qualimap` or `multiqc`, we can perform our DE analysis. That the step we will cover in depth within the Practices in Module 4. 

## Experimental planning considerations

Understanding the steps in the experimental process of RNA extraction and preparation of RNA-Seq libraries is helpful for designing an RNA-Seq experiment, but there are special considerations that should be highlighted that can greatly affect the quality of a differential expression analysis.

These important considerations include:

- Number and type of replicates
- Avoiding confounding
- Addressing batch effects

### Replicates

Experimental replicates can be performed as technical replicates or biological replicates.


- Technical replicates: use the same biological sample to repeat the technical or experimental steps in order to accurately measure technical variation and remove it during analysis.

- Biological replicates use different biological samples of the same condition to measure the biological variation between samples.

For differential expression analysis, the more biological replicates, the better the estimates of biological variation and the more precise our estimates of the mean expression levels. This leads to more accurate modeling of our data and identification of more differentially expressed genes.

*[Figure omitted: Liu, Zhou & White (2014) Bioinformatics 30:301-304, [doi:10.1093/bioinformatics/btt688](https://doi.org/10.1093/bioinformatics/btt688).]*
Image credit: [Liu, Y., et al., Bioinformatics (2014) 30(3): 301–304](https://doi.org/10.1093/bioinformatics/btt688)


As the figure above illustrates, biological replicates are of greater importance than sequencing depth, which is the total number of reads sequenced per sample. The figure shows the relationship between sequencing depth and the number of replicates on the number of differentially expressed genes identified [1]. Note that an increase in the number of replicates tends to return more DE genes than increasing the sequencing depth. Therefore, generally more replicates are better than higher sequencing depth, with the caveat that higher depth is required for detection of lowly expressed DE genes and for performing isoform-level differential expression.

Replicates are almost always preferred to greater sequencing depth for bulk RNA-Seq. However, guidelines depend on the experiment performed and the desired analysis. Below, we list some general guidelines for replicates and sequencing depth to help with experimental planning:

**General gene-level differential expression:**

- ENCODE guidelines suggest 30 million SE reads per sample (stranded).

- 15 million reads per sample is often sufficient, if there are a good number of replicates (>3).

- Spend money on more biological replicates, if possible.

- Generally recommended to have read length >= 50 bp

**Gene-level differential expression with detection of lowly-expressed genes:**

- Similarly, benefits from replicates more than sequencing depth.

- Sequence deeper with at least 30-60 million reads, depending on the level of expression (start with 30 million with a good number of replicates).

- Generally recommended to have read length >= 50 bp

**Isoform-level differential expression:**

- Of known isoforms, suggested to have a depth of at least 30 million reads per sample and paired-end reads.

- Novel isoforms should have more depth (> 60 million reads per sample).

- Choose biological replicates over paired/deeper sequencing.

- Generally recommended to have read length >= 50 bp, but longer is better as the reads will be more likely to cross exon junctions

- Perform careful QC of RNA quality. Be careful to use high-quality preparation methods and restrict analysis to high-quality RIN # samples.

**Other types of RNA analyses (intron retention, small RNA-Seq, etc.):**

- Different recommendations depending on the analysis.

- Almost always, more biological replicates are better!
---

## Setting up differential gene expression analysis

In this Module, we perform an end-to-end gene-level RNA-seq differential expression workflow using various R packages. We will start with reading in data obtained from [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275), perform exploratory data analysis for quality assessment, explore the relationship between samples, perform differential expression analysis, and visually explore the results before performing downstream functional analysis.

Adapted from [hbctraining/Intro-to-DGE](https://hbctraining.github.io/Intro-to-DGE/schedule/links-to-lessons.html)